In [20]:

import tensorflow as tf
from tensorflow.keras.models import model_from_json
import numpy as np
import os

# 檢查 TensorFlow 版本
print("TensorFlow version:", tf.__version__)

# 載入 Model 3
try:
    with open("model3.json", "r") as json_file:
        model_json = json_file.read()
    model3 = model_from_json(model_json)
    print("Model structure loaded successfully from model3.json")
except Exception as e:
    print(f"Error loading model3.json: {e}")
    print("Attempting to rebuild model manually...")
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

    # 手動重建 Model 3 結構
    model3 = Sequential([
        Conv2D(16, (3, 3), activation='relu', input_shape=(30, 30, 3), padding='valid'),
        MaxPooling2D((2, 2), strides=(2, 2), padding='valid'),
        Conv2D(32, (3, 3), activation='relu', padding='valid'),
        MaxPooling2D((2, 2), strides=(2, 2), padding='valid'),
        Conv2D(64, (3, 3), activation='relu', padding='valid'),
        MaxPooling2D((2, 2), strides=(2, 2), padding='valid'),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(43, activation='softmax')
    ])
    print("Model rebuilt manually")

# 載入權重
try:
    model3.load_weights("model3.h5")
    print("Weights loaded successfully from model3.h5")
except Exception as e:
    print(f"Error loading model3.h5: {e}")
    exit(1)

# 創建 record 目錄
if not os.path.exists("record"):
    os.makedirs("record")

# 獲取權重並儲存為 .dat
weights = model3.get_weights()
weight_names = ["w_c1", "b_c1", "w_c2", "b_c2", "w_c3", "b_c3", "w_fc1", "b_fc1", "w_fc2", "b_fc2"]
expected_shapes = [
    (3, 3, 3, 16), (16,),  # Conv1 weights, bias
    (3, 3, 16, 32), (32,),  # Conv2 weights, bias
    (3, 3, 32, 64), (64,),  # Conv3 weights, bias
    (256, 64), (64,),  # FC1 weights, bias
    (64, 43), (43,)   # FC2 weights, bias
]

for i, (w, name, exp_shape) in enumerate(zip(weights, weight_names, expected_shapes)):
    if w.shape != exp_shape:
        print(f"Warning: {name} shape {w.shape} does not match expected shape {exp_shape}")
    np.savetxt(f"record/weight_{i}.dat", w.flatten(), fmt="%.6f")
    print(f"Saved {name} to record/weight_{i}.dat, shape: {w.shape}")

print("權重匯出完成")


TensorFlow version: 2.19.0
Error loading model3.json: Could not locate class 'Sequential'. Make sure custom classes are decorated with `@keras.saving.register_keras_serializable()`. Full object config: {'class_name': 'Sequential', 'config': {'name': 'sequential_5', 'layers': [{'class_name': 'Conv2D', 'config': {'name': 'conv2d_16', 'trainable': True, 'batch_input_shape': [None, 30, 30, 3], 'dtype': 'float32', 'filters': 16, 'kernel_size': [3, 3], 'strides': [1, 1], 'padding': 'valid', 'data_format': 'channels_last', 'dilation_rate': [1, 1], 'activation': 'relu', 'use_bias': True, 'kernel_initializer': {'class_name': 'GlorotUniform', 'config': {'seed': None, 'dtype': 'float32'}}, 'bias_initializer': {'class_name': 'Zeros', 'config': {'dtype': 'float32'}}, 'kernel_regularizer': None, 'bias_regularizer': None, 'activity_regularizer': None, 'kernel_constraint': None, 'bias_constraint': None}}, {'class_name': 'MaxPooling2D', 'config': {'name': 'max_pooling2d_14', 'trainable': True, 'dtype':

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Saved w_c3 to record/weight_4.dat, shape: (3, 3, 32, 64)
Saved b_c3 to record/weight_5.dat, shape: (64,)
Saved w_fc1 to record/weight_6.dat, shape: (256, 64)
Saved b_fc1 to record/weight_7.dat, shape: (64,)
Saved w_fc2 to record/weight_8.dat, shape: (64, 43)
Saved b_fc2 to record/weight_9.dat, shape: (43,)
權重匯出完成


In [21]:

import numpy as np
from PIL import Image
import os

# 圖像預處理函數
def preprocess_image(image_path, size=30):
    # 載入圖像並轉為 RGB
    img = Image.open(image_path).convert("RGB")
    # 縮放至 30x30，使用 LANCZOS 濾波（與訓練一致）
    img = img.resize((size, size), Image.LANCZOS)
    # 轉為 numpy 陣列並正規化到 [0,1]
    img_array = np.array(img).astype("float32") / 255.0
    # 展平為一維陣列 (NHWC 格式)
    img_flat = img_array.flatten()
    return img_flat

# 主程式
def main():
    # 設定圖像路徑（替換為你的測試圖像）
    image_path = "00001.png"  # 示例圖像
    if not os.path.exists(image_path):
        print(f"Error: Image file {image_path} not found")
        return

    # 預處理圖像
    input_image = preprocess_image(image_path, size=30)

    # 檢查尺寸
    expected_size = 30 * 30 * 3
    if len(input_image) != expected_size:
        print(f"Error: Input image size {len(input_image)} does not match expected {expected_size}")
        return

    # 創建 record 目錄
    if not os.path.exists("record"):
        os.makedirs("record")

    # 儲存為 input_image.dat
    output_path = "record/input_image.dat"
    np.savetxt(output_path, input_image, fmt="%.6f")
    print(f"Input image saved to {output_path}, size: {len(input_image)}")

if __name__ == "__main__":
    main()


Input image saved to record/input_image.dat, size: 2700


## Try


In [12]:
# ===========================
# GTSRB CNN in TensorFlow 2 / Keras
# 保留原本 CNN 架構，只改成 43 類 (交通標誌分類)
# ===========================

import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import pandas as pd
import os
from PIL import Image

# ===========================
# Google Drive 路徑
# ===========================
gtsrb_root = "/content/drive/MyDrive/Colab Notebooks/GTSRB"
train_csv = os.path.join(gtsrb_root, "Train.csv")
test_csv  = os.path.join(gtsrb_root, "Test.csv")
train_dir = os.path.join(gtsrb_root, "")
test_dir  = os.path.join(gtsrb_root, "")

# ===========================
# 載入 GTSRB 資料集
# ===========================
def load_gtsrb(csv_file, root_dir, img_size=(32,32)):
    df = pd.read_csv(csv_file)
    X, y = [], []
    for i, row in df.iterrows():
        img_path = os.path.join(root_dir, row['Path'])
        img = Image.open(img_path).resize(img_size)
        img = np.array(img).astype(np.float32) / 255.0
        X.append(img)
        y.append(row['ClassId'])
    return np.array(X), np.array(y)

print("Loading dataset...")
x_train, y_train = load_gtsrb(train_csv, train_dir)
x_test, y_test   = load_gtsrb(test_csv, test_dir)

num_classes = 43
print("Train:", x_train.shape, y_train.shape)
print("Test :", x_test.shape, y_test.shape)

# One-hot 編碼
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test  = tf.keras.utils.to_categorical(y_test, num_classes)

# ===========================
# 定義 CNN 模型
# ===========================
model = models.Sequential()

# 第一個卷積層 + 池化
model.add(layers.Conv2D(32, (5,5), activation='relu', padding='valid', input_shape=(32,32,3)))
model.add(layers.MaxPooling2D((2,2)))

# 第二個卷積層 + 池化
model.add(layers.Conv2D(64, (5,5), activation='relu', padding='valid'))
model.add(layers.MaxPooling2D((2,2)))

# 第三個卷積層
model.add(layers.Conv2D(128, (3,3), activation='relu', padding='valid'))
model.add(layers.Flatten())

# 全連接層
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dropout(0.5))

# 輸出層 (改成 43 類)
model.add(layers.Dense(num_classes, activation='softmax'))

model.summary()

# ===========================
# 編譯模型
# ===========================
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# ===========================
# 訓練模型
# ===========================
history = model.fit(x_train, y_train,
                    epochs=50,        # GTSRB 比 CIFAR-10 複雜，可以先設 50
                    batch_size=64,
                    validation_split=0.1)

# ===========================
# 評估模型
# ===========================
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test accuracy:", test_acc)

# ===========================
# 導出權重為 .dat 檔案 (給 HLS 用)
# ===========================
if not os.path.exists('record'):
    os.makedirs('record')

weights = model.get_weights()
for i, w in enumerate(weights):
    np.savetxt(f'record/weight_{i}.dat', w.flatten(), fmt='%0.6f')
print("權重已保存到 record/ 目錄")


Loading dataset...


KeyboardInterrupt: 

In [7]:
import pandas as pd

# Load the CSV file and print the column names to identify the correct filename column
csv_file_path = "/content/drive/MyDrive/Colab Notebooks/GTSRB/Train.csv"
try:
    df = pd.read_csv(csv_file_path)
    print("Columns in the CSV file:")
    print(df.columns)
except FileNotFoundError:
    print(f"Error: File not found at {csv_file_path}")

Columns in the CSV file:
Index(['Width', 'Height', 'Roi.X1', 'Roi.Y1', 'Roi.X2', 'Roi.Y2', 'ClassId',
       'Path'],
      dtype='object')
